In [ ]:
#Google Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.utils import shuffle
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

SEED = 13
N_SPLITS = 5

DATA_DIR = Path("/content/drive/My Drive/Colab Notebooks/TFM/DataSet")
INPUT_PATH = DATA_DIR / "02_only_speech.csv"
PARTITIONS_PATH = DATA_DIR / "data_partitions_paper_ready.csv"
OUT_DIR = DATA_DIR / "02_results_speech_xgboost_paper"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", INPUT_PATH)
print("Partitions:", PARTITIONS_PATH)
print("Output:", OUT_DIR)

Input: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/02_only_speech.csv
Partitions: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/data_partitions_paper_ready.csv
Output: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/02_results_speech_xgboost_paper


In [3]:
def get_metrics(y_true, y_pred, y_prob):
    return {
        "WAcc": accuracy_score(y_true, y_pred),
        "UAcc": balanced_accuracy_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }


def evaluate_subject_level(pred_conv):
    # Media de probabilidades por sujeto para pasar de conversación a sujeto.
    pred_subject = (
        pred_conv
        .groupby(["subject_id", "label", "outer_fold"], as_index=False)["prob_1"]
        .mean()
    )
    pred_subject["pred"] = (pred_subject["prob_1"] >= 0.5).astype(int)
    return pred_subject, get_metrics(
        pred_subject["label"],
        pred_subject["pred"],
        pred_subject["prob_1"],
    )

In [4]:
data = pd.read_csv(INPUT_PATH)
partitions = pd.read_csv(PARTITIONS_PATH)

feature_cols = sorted(
    [col for col in data.columns if col.startswith("speech_")],
    key=lambda col: int(col.split("_", 1)[1]),
)

required = {"subject_id", "avatar", "label"}
if not required.issubset(data.columns):
    raise ValueError(f"Faltan columnas en {INPUT_PATH.name}: {required - set(data.columns)}")
if not {"subject_id", "avatar", "outer_fold"}.issubset(partitions.columns):
    raise ValueError("data_partitions_paper_ready.csv debe tener subject_id, avatar y outer_fold")

# El fichero de particiones ya viene preparado con los mismos nombres de avatar que el CSV de embeddings.
partitions = shuffle(partitions, random_state=SEED).reset_index(drop=True)

df = data.merge(
    partitions[["subject_id", "avatar", "outer_fold"]],
    on=["subject_id", "avatar"],
    how="inner",
)

if len(df) != len(data):
    missing = len(data) - len(df)
    raise ValueError(f"Hay {missing} filas del CSV de embeddings sin partición externa")

print("Filas:", len(df))
print("Sujetos:", df["subject_id"].nunique())
print("Variables:", len(feature_cols))
print("Distribución de folds:")
display(df.groupby("outer_fold")["subject_id"].nunique().to_frame("n_subjects"))

Filas: 600
Sujetos: 101
Variables: 1024
Distribución de folds:


,n_subjects
outer_fold,
1,21
2,20
3,20
4,20
5,20


In [5]:
param_grid = {
    "max_depth": list(range(3, 12)),
    "n_estimators": [25, 50, 100, 200],
}

scoring = {
    "WAcc": "accuracy",
    "UAcc": "balanced_accuracy",
    "auc": "roc_auc",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
}

all_conv_predictions = []
conv_metrics_rows = []
subject_metrics_rows = []
best_params_rows = []

for fold in sorted(df["outer_fold"].unique()):
    print(f"===== OUTER FOLD {fold} =====")

    dev = df[df["outer_fold"] != fold].reset_index(drop=True)
    test = df[df["outer_fold"] == fold].reset_index(drop=True)

    X_dev = dev[feature_cols].to_numpy()
    y_dev = dev["label"].to_numpy()
    g_dev = dev["subject_id"].to_numpy()

    X_test = test[feature_cols].to_numpy()
    y_test = test["label"].to_numpy()

    inner_cv = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED,
    )

    grid = GridSearchCV(
        estimator=XGBClassifier(),
        param_grid=param_grid,
        scoring=scoring,
        refit="UAcc",
        cv=inner_cv,
        n_jobs=-1,
        verbose=0,
    )
    grid.fit(X_dev, y_dev, groups=g_dev)

    best_params = grid.best_params_
    best_idx = grid.best_index_
    best_inner_f1 = grid.cv_results_["mean_test_f1"][best_idx]
    best_inner_uacc = grid.cv_results_["mean_test_UAcc"][best_idx]

    model = XGBClassifier(**best_params)
    model.fit(X_dev, y_dev)

    prob_1 = model.predict_proba(X_test)[:, 1]
    pred = (prob_1 >= 0.5).astype(int)

    pred_conv = test[["subject_id", "avatar", "label", "outer_fold"]].copy()
    pred_conv["prob_1"] = prob_1
    pred_conv["pred"] = pred
    all_conv_predictions.append(pred_conv)

    conv_metrics = get_metrics(y_test, pred, prob_1)
    conv_metrics["outer_fold"] = fold
    conv_metrics_rows.append(conv_metrics)

    pred_subject, subject_metrics = evaluate_subject_level(pred_conv)
    subject_metrics["outer_fold"] = fold
    subject_metrics_rows.append(subject_metrics)

    best_params_rows.append({
        "outer_fold": fold,
        "best_max_depth": best_params["max_depth"],
        "best_n_estimators": best_params["n_estimators"],
        "best_inner_UAcc": best_inner_uacc,
        "best_inner_f1": best_inner_f1,
    })

    print("Best params:", best_params)
    print("Conversation-level F1:", round(conv_metrics["f1"], 3))
    print("Subject-level F1:", round(subject_metrics["f1"], 3))

===== OUTER FOLD 1 =====
Best params: {'max_depth': 9, 'n_estimators': 100}
Conversation-level F1: 0.447
Subject-level F1: 0.571
===== OUTER FOLD 2 =====
Best params: {'max_depth': 10, 'n_estimators': 100}
Conversation-level F1: 0.505
Subject-level F1: 0.556
===== OUTER FOLD 3 =====
Best params: {'max_depth': 5, 'n_estimators': 25}
Conversation-level F1: 0.372
Subject-level F1: 0.462
===== OUTER FOLD 4 =====
Best params: {'max_depth': 9, 'n_estimators': 25}
Conversation-level F1: 0.64
Subject-level F1: 0.75
===== OUTER FOLD 5 =====
Best params: {'max_depth': 10, 'n_estimators': 50}
Conversation-level F1: 0.574
Subject-level F1: 0.571


In [7]:
conv_metrics_df = pd.DataFrame(conv_metrics_rows)
subject_metrics_df = pd.DataFrame(subject_metrics_rows)
best_params_df = pd.DataFrame(best_params_rows)
predictions_df = pd.concat(all_conv_predictions, ignore_index=True)

global_subject_predictions, global_subject_metrics = evaluate_subject_level(predictions_df)

conv_metrics_df.to_csv(OUT_DIR / "conversation_level_outer_metrics.csv", index=False)
subject_metrics_df.to_csv(OUT_DIR / "subject_level_outer_metrics.csv", index=False)
best_params_df.to_csv(OUT_DIR / "best_params_by_outer_fold.csv", index=False)
predictions_df.to_csv(OUT_DIR / "conversation_predictions.csv", index=False)
global_subject_predictions.to_csv(OUT_DIR / "subject_predictions_global.csv", index=False)

print("Conversation-level Test metrics: mean ± std")
display(pd.concat([
    conv_metrics_df[["WAcc", "UAcc", "auc", "f1", "precision", "recall", "kappa"]].mean().round(3).rename("mean"),
    conv_metrics_df[["WAcc", "UAcc", "auc", "f1", "precision", "recall", "kappa"]].std().round(3).rename("std"),
], axis=1))

print("Subject-level Test metrics: mean ± std")
display(pd.concat([
    subject_metrics_df[["WAcc", "UAcc", "auc", "f1", "precision", "recall", "kappa"]].mean().round(3).rename("mean"),
    subject_metrics_df[["WAcc", "UAcc", "auc", "f1", "precision", "recall", "kappa"]].std().round(3).rename("std"),
], axis=1))

print("Subject-level global metrics")
display(pd.Series(global_subject_metrics).round(3).to_frame("global"))

print("Best params by outer fold")
display(best_params_df)

print("Archivos guardados en:", OUT_DIR)

Conversation-level Test metrics: mean ± std


,mean,std
WAcc,0.623,0.065
UAcc,0.603,0.072
auc,0.659,0.081
f1,0.508,0.105
precision,0.551,0.082
recall,0.481,0.141
kappa,0.207,0.142


Subject-level Test metrics: mean ± std


,mean,std
WAcc,0.693,0.075
UAcc,0.668,0.079
auc,0.702,0.107
f1,0.582,0.104
precision,0.674,0.102
recall,0.525,0.142
kappa,0.346,0.155


Subject-level global metrics


,global
WAcc,0.693
UAcc,0.669
auc,0.708
f1,0.587
precision,0.667
recall,0.524
kappa,0.348


Best params by outer fold


,outer_fold,best_max_depth,best_n_estimators,best_inner_UAcc,best_inner_f1
0,1,9,100,0.603433,0.518548
1,2,10,100,0.624821,0.533276
2,3,5,25,0.580868,0.465313
3,4,9,25,0.559947,0.434329
4,5,10,50,0.555323,0.407216


Archivos guardados en: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/02_results_speech_xgboost_paper
